In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

splits = pd.read_csv(r"D:\AgriVision\data\splits.csv")
train_df = splits[splits.split == "train"]
val_df = splits[splits.split == "val"]
test_df = splits[splits.split == "test"]

classes = sorted(splits["label"].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}

IMG_SIZE, BATCH = 224, 32

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return img, label   # no /255 here: MobileNetV2 preprocessing is inside the model

def make_ds(frame, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (frame["path"].values, frame["label"].map(class_to_idx).values))
    if shuffle:
        ds = ds.shuffle(len(frame), seed=42)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df), make_ds(test_df)

y_train = train_df["label"].map(class_to_idx).values
w = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(w))
print(classes)

['BrownRust', 'Healthy', 'Mildew', 'Septoria', 'YellowRust']


In [2]:
from tensorflow import keras
from tensorflow.keras import layers

augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
])

base = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
base.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = augment(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(classes), activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [3]:
history = model.fit(train_ds, validation_data=val_ds, epochs=10,
                    class_weight=class_weights)

Epoch 1/10


d:\AgriVision\venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


22/22 ━━━━━━━━━━━━━━━━━━━━ 73s 3s/step - accuracy: 0.2632 - loss: 1.7135 - val_accuracy: 0.4467 - val_loss: 1.3534
Epoch 2/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 51s 2s/step - accuracy: 0.4177 - loss: 1.3460 - val_accuracy: 0.5467 - val_loss: 1.1725
Epoch 3/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 54s 2s/step - accuracy: 0.5093 - loss: 1.1918 - val_accuracy: 0.5667 - val_loss: 1.1528
Epoch 4/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 71s 2s/step - accuracy: 0.5322 - loss: 1.0596 - val_accuracy: 0.5933 - val_loss: 1.0535
Epoch 5/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 44s 2s/step - accuracy: 0.5651 - loss: 0.9970 - val_accuracy: 0.6000 - val_loss: 1.0216
Epoch 6/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 83s 2s/step - accuracy: 0.6009 - loss: 0.9582 - val_accuracy: 0.6467 - val_loss: 0.9429
Epoch 7/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 98s 3s/step - accuracy: 0.6309 - loss: 0.9135 - val_accuracy: 0.6333 - val_loss: 0.9555
Epoch 8/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 59s 3s/step - accuracy: 0.6466 - loss: 0.8440 - val_accuracy: 0.6600 - val_loss: 0.8864
Epo

In [4]:
loss, acc = model.evaluate(test_ds)
print("Test accuracy:", acc)
model.save(r"D:\AgriVision\models\leaf_model.keras")

5/5 ━━━━━━━━━━━━━━━━━━━━ 7s 1s/step - accuracy: 0.6600 - loss: 0.9382
Test accuracy: 0.6600000262260437


In [ ]:
base.trainable = True
for layer in base.layers[:-30]:      # keep the early layers frozen
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

early = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3,
                                      restore_best_weights=True)

history_ft = model.fit(train_ds, validation_data=val_ds, epochs=15,
                       class_weight=class_weights, callbacks=[early])

loss, acc = model.evaluate(test_ds)
print("Test accuracy after fine-tuning:", acc)
model.save(r"D:\AgriVision\models\leaf_model_ft.keras")

Epoch 1/15
14/22 ━━━━━━━━━━━━━━━━━━━━ 25s 3s/step - accuracy: 0.5714 - loss: 1.1682

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

y_true, y_pred = [], []
for x, y in test_ds:
    p = model.predict(x, verbose=0)
    y_true.extend(y.numpy())
    y_pred.extend(np.argmax(p, axis=1))

print(classification_report(y_true, y_pred, target_names=classes))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion matrix (test set)")
plt.show()